# 06 - LangGraph agent: the state machine

Now assemble the whole pipeline (fetch -> classify -> detect trend
-> drill down -> summarise) as a **LangGraph state machine** with a
conditional edge. This is where LangGraph earns its place over a
linear LangChain chain: the drill-down branch only fires when a
category's share swings by more than a threshold vs the previous
window, and the graph itself decides which categories to zoom
into.

**What you will learn**

- The LangGraph state/node/edge model: how it differs from a
  linear LangChain chain and when the difference matters.
- How to define a `TypedDict` state that survives serialisation
  between nodes.
- How to add a **conditional edge** that routes to a drill-down
  sub-node only when a subtopic's share actually spikes.
- How to visualise and debug the graph while it is running.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage

from src.llm import get_chat_ollama
from src.classify import LLMClassifier
from src.trend import category_deltas, top_trending
from src.labels import CATEGORIES, category_names
from src.agent import AgentState

pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 100)

## 1. Set up the two windows

Trend detection compares a *current window* of stories against a
*previous window*. For this demo we use the 223 AI/ML stories we
already fetched: split them in half by story age. The younger half
is "current"; the older half is "previous". No new API calls, no
network dependency for reproducibility.

In [2]:
stories = pd.read_csv(ROOT / "data" / "stories_ai.csv")
silver = pd.read_csv(ROOT / "data" / "silver_labels.csv")

merged = stories.merge(silver, on="id", how="inner")
merged["title"] = merged["title"].fillna("")
merged["text"] = merged["text"].fillna("")
merged["created_at"] = pd.to_datetime(merged["created_at"])
merged = merged.sort_values("created_at").reset_index(drop=True)

# Split by median age: younger half = "current", older half = "previous"
mid = len(merged) // 2
previous_window = merged.iloc[:mid].copy()
current_window  = merged.iloc[mid:].copy()

print(f"previous window: {len(previous_window):>3} stories  "
      f"({previous_window['created_at'].min()}  ->  {previous_window['created_at'].max()})")
print(f"current  window: {len(current_window):>3} stories  "
      f"({current_window['created_at'].min()}   ->  {current_window['created_at'].max()})")

previous window: 111 stories  (2026-08-12 15:01:17+00:00  ->  2026-08-19 14:23:13+00:00)
current  window: 112 stories  (2026-08-19 14:26:42+00:00   ->  2026-08-19 20:30:04+00:00)


## 2. Define the LangGraph state

A `TypedDict` because LangGraph serialises state between nodes. We
keep the payload minimal: two windows (as DataFrames), the
detected trend deltas, the drill-down categories flagged, and the
final report.

In [3]:
class TrendState(TypedDict, total=False):
    previous: pd.DataFrame     # older-half stories, already categorised
    current:  pd.DataFrame     # newer-half stories, gets classified in node_classify
    trend:    pd.DataFrame     # per-category delta table from node_detect_trend
    drill:    list[str]        # category names flagged for drill-down
    per_cat_summary: dict      # per-category short summary text
    report:   str              # final rendered trend report

DRILL_THRESHOLD = 0.03   # 3 percentage-point swing
TOP_N = 3

print(f"drill-down threshold: {DRILL_THRESHOLD * 100:.1f} percentage-point swing")
print(f"top N spiking categories to drill into: {TOP_N}")

drill-down threshold: 3.0 percentage-point swing
top N spiking categories to drill into: 3


## 3. Define the nodes

Each node is a pure function `state -> state`. LangGraph handles
routing between them.

- **node_classify**: use the LLM classifier on the current window.
  We already have silver labels for the previous window from nb02.
- **node_detect_trend**: compute per-category share deltas.
- **node_should_drill_down**: (conditional edge) route to drill-down
  or straight to summarise based on whether any category swings
  > threshold.
- **node_drill_down**: for each flagged category, produce a short
  LLM summary of what the current-window stories in that category
  are about.
- **node_summarise**: assemble the final human-readable report.

In [ ]:
chat = get_chat_ollama(model="llama3.1:8b")
clf = LLMClassifier(chat=chat)


def node_classify(state: TrendState) -> TrendState:
    cur = state["current"].copy()
    preds = clf.predict(cur["title"].tolist(), cur["text"].tolist())
    cur["llm_category"] = preds
    # We use silver on prev + LLM on cur; both are per-story categories.
    return {"current": cur}


def node_detect_trend(state: TrendState) -> TrendState:
    # The previous window keeps its silver ``category`` column.
    # The current window has both silver ``category`` (from load-time
    # merge) and the LLM-produced ``llm_category`` (from
    # ``node_classify``); use the LLM one for trend detection.
    prev = state["previous"][["category"]].copy()
    cur = state["current"][["llm_category"]].rename(
        columns={"llm_category": "category"},
    )
    freq = category_deltas(cur, prev, weight_col=None)
    return {"trend": freq}


def node_should_drill_down(state: TrendState) -> str:
    trend = state["trend"]
    rising = trend[trend["delta"] >= DRILL_THRESHOLD]
    if len(rising) == 0:
        return "summarise"
    return "drill_down"


def _summarise_titles(cat: str, titles: list[str]) -> str:
    if not titles:
        return f"(no {cat} stories to summarise)"
    joined = "\n".join(f"- {t}" for t in titles[:12])
    prompt = (
        f"You are producing a one-paragraph summary of what is "
        f"trending on Hacker News in the '{cat}' category this "
        f"window. Read the story titles below and write a single "
        f"paragraph (2-4 sentences) describing the common threads "
        f"and any specific standout items. Do not quote titles "
        f"verbatim; describe themes.\n\nTitles:\n{joined}\n\n"
        f"Paragraph:"
    )
    resp = chat.invoke([HumanMessage(content=prompt)])
    return resp.content.strip()


def node_drill_down(state: TrendState) -> TrendState:
    trend = state["trend"]
    cur = state["current"]
    rising = trend[trend["delta"] >= DRILL_THRESHOLD].sort_values("delta", ascending=False)
    drill = rising.head(TOP_N)["category"].tolist()

    per_cat = {}
    for cat in drill:
        titles = cur.loc[cur["llm_category"] == cat, "title"].tolist()
        per_cat[cat] = _summarise_titles(cat, titles)
    return {"drill": drill, "per_cat_summary": per_cat}


def node_summarise(state: TrendState) -> TrendState:
    trend = state["trend"]
    lines = ["# HN AI/ML trend report", ""]
    lines.append(f"- previous window: {len(state['previous'])} stories")
    lines.append(f"- current window:  {len(state['current'])} stories")
    lines.append("")
    lines.append("## Category share deltas (current vs previous)")
    lines.append("")
    for _, row in trend.iterrows():
        arrow = "up" if row["delta"] >= 0 else "down"
        lines.append(
            f"- **{row['category']}**: {row['current_share'] * 100:.1f}% "
            f"({arrow} {abs(row['delta']) * 100:.1f} pp from "
            f"{row['previous_share'] * 100:.1f}%)"
        )
    if state.get("per_cat_summary"):
        lines.append("")
        lines.append("## Drill-down summaries")
        for cat, para in state["per_cat_summary"].items():
            lines.append("")
            lines.append(f"### {cat}")
            lines.append("")
            lines.append(para)
    return {"report": "\n".join(lines)}

## 4. Assemble the graph

In [5]:
graph = StateGraph(TrendState)
graph.add_node("classify", node_classify)
graph.add_node("detect_trend", node_detect_trend)
graph.add_node("drill_down", node_drill_down)
graph.add_node("summarise", node_summarise)

graph.set_entry_point("classify")
graph.add_edge("classify", "detect_trend")
graph.add_conditional_edges(
    "detect_trend",
    node_should_drill_down,
    {"drill_down": "drill_down", "summarise": "summarise"},
)
graph.add_edge("drill_down", "summarise")
graph.add_edge("summarise", END)

app = graph.compile()
print("graph compiled")
print()
print("nodes:", list(graph.nodes))

graph compiled

nodes: ['classify', 'detect_trend', 'drill_down', 'summarise']


## 5. Visualise the graph

LangGraph can render itself as an ASCII diagram or a mermaid
graph. ASCII is safest here (no network required for the
mermaid renderer).

In [6]:
print(app.get_graph().draw_ascii())

             +-----------+        
             | __start__ |        
             +-----------+        
                   *              
                   *              
                   *              
             +----------+         
             | classify |         
             +----------+         
                   *              
                   *              
                   *              
           +--------------+       
           | detect_trend |       
           +--------------+       
            ...          ..       
           .               ..     
         ..                  ..   
+------------+                 .. 
| drill_down |               ..   
+------------+             ..     
            ***          ..       
               *       ..         
                **   ..           
             +-----------+        
             | summarise |        
             +-----------+        
                   *              
                   *

## 6. Run the agent

In [7]:
initial_state: TrendState = {
    "previous": previous_window,
    "current":  current_window,
}
final_state = app.invoke(initial_state)

print("nodes visited: classify -> detect_trend -> "
      + ("drill_down -> " if final_state.get('drill') else "")
      + "summarise")
print()
if final_state.get("drill"):
    print(f"drill-down fired on categories: {final_state['drill']}")
else:
    print("no category exceeded the drill-down threshold this window")

nodes visited: classify -> detect_trend -> drill_down -> summarise

drill-down fired on categories: ['research', 'tool']


## 7. Inspect the trend table + drill-down summaries

In [8]:
print(final_state["trend"].to_string(index=False))

category  current_share  previous_share     delta    ratio
research       0.232143        0.099099  0.133044 2.342532
 opinion       0.151786        0.261261 -0.109476 0.580973
    tool       0.133929        0.099099  0.034829 1.351461
 product       0.223214        0.252252 -0.029038 0.884885
tutorial       0.089286        0.108108 -0.018822 0.825893
    news       0.160714        0.171171 -0.010457 0.938910
   other       0.008929        0.009009 -0.000080 0.991071


In [9]:
if final_state.get("per_cat_summary"):
    for cat, para in final_state["per_cat_summary"].items():
        print(f"### {cat}")
        print(para)
        print()
else:
    print("(no drill-down summaries this run)")

### research
The current trending topics on Hacker News in the 'research' category revolve around advancements and applications of Artificial Intelligence, particularly Large Language Models (LLMs). Several stories highlight the increasing portability and versatility of AI agents, with some even being used to create art, such as self-portraits. Researchers are also exploring ways to improve LLMs, including prompt caching and using quantum data for teaching. Additionally, there is a focus on verifying the authenticity of AI-generated content, including watermarked text, which raises questions about ownership and accountability in the age of AI.

### tool
The current trending topics on Hacker News in the 'tool' category revolve around AI and machine learning, with a focus on making these technologies more accessible and user-friendly. Several submissions showcase tools for packaging and running large language models (LLMs) locally, such as self-contained boxes and local AI tools that can

## 8. The rendered report

Everything above assembled into a single markdown document
suitable for a weekly digest email or a status channel post.

In [10]:
print(final_state["report"])

# HN AI/ML trend report

- previous window: 111 stories
- current window:  112 stories

## Category share deltas (current vs previous)

- **research**: 23.2% (up +13.3 pp from 9.9%)
- **opinion**: 15.2% (down 10.9 pp from 26.1%)
- **tool**: 13.4% (up +3.5 pp from 9.9%)
- **product**: 22.3% (down 2.9 pp from 25.2%)
- **tutorial**: 8.9% (down 1.9 pp from 10.8%)
- **news**: 16.1% (down 1.0 pp from 17.1%)
- **other**: 0.9% (down 0.0 pp from 0.9%)

## Drill-down summaries

### research

The current trending topics on Hacker News in the 'research' category revolve around advancements and applications of Artificial Intelligence, particularly Large Language Models (LLMs). Several stories highlight the increasing portability and versatility of AI agents, with some even being used to create art, such as self-portraits. Researchers are also exploring ways to improve LLMs, including prompt caching and using quantum data for teaching. Additionally, there is a focus on verifying the authenticity of 

## 9. Persist the report for nb07

In [11]:
out = ROOT / "data" / "trend_report.md"
out.write_text(final_state["report"], encoding="utf-8")
print(f"saved trend report to {out}  ({len(final_state['report'])} chars)")

saved trend report to C:\Users\anjan\Desktop\Goals\github\hn-ml-trends\data\trend_report.md  (1837 chars)


## Takeaways for notebook 07

- The graph in this notebook has one conditional edge; that is
  the smallest example that justifies LangGraph over a linear
  LangChain chain. Bigger real-world graphs branch on retrieval
  quality, user intent, tool call outcomes, etc.
- The drill-down node calls the LLM once per flagged category,
  each with only ~12 titles as context. That is what makes it
  affordable to zoom in without re-embedding everything.
- Notebook 07 evaluates whether the trend detector's top-N
  categories are the ones a human reader would agree are actually
  "trending" this window, and compares the generated report
  against a single-prompt-baseline summary.